# Advanced Text Generation Techniques and Tools

We will continue on the thoughts of prompt engineering. What can we do to further enhance the experience and output that we get from the LLM without needing to fine-tune the model itself? Fortunately, a great deal of methods and techniques allow us to further improve what we started in notebook 6. These more advanced techniques lie at the foundation of numerous LLM-focused systems and are, arguably, one of the first things users implement when designing such systems. We will explore: 

- ***Model I/O***: Loading and working with LLMs
- ***Memory***: Helping LLMs to remember
- ***Agents***: Combining complex behavior with external tools
- ***Chains***: Connecting methods and modules

These methods are all integrated with the `LangChain` framework that will help us easily use these advanced techniques throughout this chapter. `LangChain` is one of the earlier frameworks that simplify working with LLMs through useful abstractions. Newer frameworks of note are `DSPy` and `Haystack`. Some of these abstractions are illustrated below. Note that retrieval will be discussed in the next chapter.

![Alt text](./figures/langchain.png)

## Model I/O: Loading Quantized Models with LangChain

As in previous chapters, we will be using Phi-3 but with a twist; we will use a GGUF model variant instead. A GGUF model represents a compressed version of its original counterpart through a method called quantization, which reduces the number of bits needed to represent the parameters of an LLM. Bits, a series of 0s and 1s, represent values by encoding them in binary form. More bits result in a wider range of values but requires more memory to store those values, as shown

![Alt text](./figures/32-bit-16bit.png)

Quantization reduces the number of bits required to represent the parameters of an LLM while attempting to maintain most of the original information. This comes with some loss in precision but often makes up for it as the model is much faster to run, requires less VRAM, and is often almost as accurate as the original. In notebook 12, we will further discuss how quantization works under the hood. You can also see a full visual guide to quantization in “A Visual Guide to Quantization” by Maarten Grootendorst. For now, it is important to know that we will use an 8-bit variant of Phi-3 compared to the original 16-bit variant, cutting the memory requirements almost in half.

In [2]:
# download the model
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [1]:
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048, 
    seed=42,
    verbose=False, 
    stop=["<|end|>"]
)

In [2]:
# We use the invoke function to generate output
llm.invoke("Hi! My name is Bamz. What is 2 + 2?")

''

Unfortunately, we get no output! As we have seen in previous chapters, Phi-3 requires a specific prompt template. Compared to our examples with transformers, we will need to explicitly use a template ourselves. Instead of copy-pasting this template each time we use Phi-3 in LangChain, we can use one of LangChain’s core functionalities, namely “chains.”

## Chains: Extending the Capabilitie of LLMs

LangChain is named after one of its main methods, chains. Although we can run LLMs in isolation, their power is shown when used with additional components or even when used in conjunction with each other. Chains not only allow for extending the capabilities of LLMs but also for multiple chains to be connected together. The most basic form of a chain in LangChain is a single chain. Although a chain can take many forms, each with a different complexity, it generally connects an LLM with some additional tool, prompt, or feature.

![Alt text](./figures/single-chain.png)

In practice, chains can become complex quite quickly. We can extend the prompt template however we want and we can even combine several separate chains together to create intricate systems. In order to thoroughly understand what is happening in a chain, let’s explore how we can add Phi-3’s prompt template to the LLM.

### A Single Link in the Chain: Prompt Template

We start with creating our first chain, namely the prompt template that Phi-3 expects. In the previous chapter, we explored how `transformers.pipeline` applies the chat template automatically. This is not always the case with other packages and they might need the prompt template to be explicitly defined. With LangChain, we will use chains to create and use a default prompt template. It also serves as a nice hands-on experience with using prompt templates. The idea is illustrated below, 


![Alt text](./figures/prompt-chain.png)

The template for Phi-3 is compromised of four main components: 
* `<s>` to indicate when the prompt starts
* `<|user|>` to indicate the start of the user’s prompt
* `<|assistant|>` to indicate the start of the model’s output
* `<|end|>` to indicate the end of either the prompt or the model’s output

Using this template, the model takes in a system_prompt, which generally describes what we expect from the LLM. Then, we can use the input_prompt to ask the LLM specific questions.


In [3]:
from langchain import PromptTemplate

# Create a prompt template with the input_prompt variable
template = """<|user|>
{input_prompt}<|end|>
<|assistant|>
"""
prompt = PromptTemplate(
    template=template,
    input_variables=['input_prompt']
)

In [6]:
# lets create our first chain
# we 'chain' the prompt we created with the llm
basic_chain = prompt | llm 

In [7]:
# Use the chain 
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Bamz. What is 2+ 2?"
    }
)

" Hello Bamz! The answer to 2 + 2 is 4.\n\nHere's a simple calculation:\n\n1. Start with the number 2.\n2. Add another 2 to it.\n3. You get the total, which is 4.\n\nThat's basic arithmetic!"

In [14]:
# Lets create a chain that creates our business' name 
template = "Create a funny name for a business that sells {product}."
name_prompt = PromptTemplate(
    template=template, 
    input_variables=['product']
)
funny_business_name_chain = name_prompt | llm 
funny_business_name_chain.invoke(
    {
        "product": "football"
    }
)

'\n<|assistant|> "Kickin\' Goals Galore - Your One Stop Shop for All Things Football!" \n\nOr, how about this:\n\n"Football Frenzy Emporium - Where Every Pitch is a Party'

Adding a prompt template to the chain is just the very first step you need to enhance the capabilities of your LLM. Throughout this chapter, we will see many ways in which we can add additional modular components to existing chains, starting with memory.

### A Chain with Multiple Prompts

In our previous example, we created a single chain consisting of a prompt template and an LLM. Since our example was quite straightforward, the LLM had no issues dealing with the prompt. However, some applications are more involved and require lengthy or complex prompts to generate a response that captures those intricate details. Instead, we could break this complex prompt into smaller subtasks that can be run sequentially. This would require multiple calls to the LLM but with smaller prompts and intermediate outputs as shown,

![Alt text](./figures/multiple-prompts.png)

This process of using multiple prompts is an extension of our previous example. Instead of using a single chain, we link chains where each link deals with a specific subtask. 
Let’s illustrate with an example. Assume that we want to generate a story that has three components:

* A title
* A description of the main character
* A summary of the story
  
Instead of generating everything in one go, we create a chain that only requires a single input by the user and then sequentially generates the three components.

![Alt text](./figures/story-chain.png)

To generate that story, we use LangChain to describe the first component, namely the title. This first link is the only component that requires some input from the user. We define the template and use the `"summary"` variable as the input variable and `"title"` as the output.


In [4]:
from langchain import LLMChain

# Create a chain for the title of our story
template="""<|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>
"""
title_prompt = PromptTemplate(
    template=template, 
    input_variables=['summary']
)
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title") # this method will be deprecated

/var/folders/lt/dwt7vtzj1sx2nvf4fgrn8r_00000gn/T/ipykernel_10472/3310691214.py:12: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title") # this method will be deprecated


In [47]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Mother\'s Love: A Heartrending Journey"'}

In [48]:
# Generate next component
# create a chain for the character description using the summary
template = """<|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>
"""
character_prompt = PromptTemplate(
    template=template, input_variables=['summary', 'title']
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [49]:
# Although we can use it to generate the character it will be used as part of the automated chain instead.
# Create chain for the story using the summary, title, and character description. 
template = """<|user|>
Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph. <|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=['summary', 'title', 'character']
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [50]:
# Now combine all three
llm_chain = title | character | story
llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of Love: A Tale of Loss and Rediscovery"',
 'character': " The protagonist, Maya, is a resilient and introspective teenager who struggles to cope with the devastating loss of her beloved mother. As she navigates through grief and self-discovery, Maya learns to find comfort in cherished memories and newfound strengths that ultimately guide her on a journey towards healing and acceptance.\n\nMaya's tender-hearted nature is shaped by the profound impact of losing her mother at a young age, yet she remains determined to uncover deeper truths about love, connection, and herself as she embarks on this poignant and transformative journey through loss and rediscovery.",
 'story': ' "Whispers of Love: A Tale of Loss and Rediscovery" follows Maya, a resilient teenager who experiences profound grief when her cherished mother passes away unexpectedly. Haunted by memories yet determined to honor her mother\'s love, Maya embarks on an 

## Memory: Helping LLMs to Remember Conversations

When we are using LLMs out of the box, they will not remember what was being said in a conversation. You can share your name in one prompt but it will have forgotten it by the next prompt.

In [51]:
# Lets give the LLM our name 
basic_chain.invoke({"input_prompt": "Hi! My name is Bamz! Whats 1 + 1?"})

" Hello Bamz! The answer to 1 + 1 is 2.\n\nHere's a simple math breakdown:\n- You start with the number 1.\n- Then you add another 1 to it.\n- When you combine them, you get 2 as the total sum."

In [52]:
# Lets reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" As an AI, I'm unable to determine your name without prior information provided by you. If you'd like to share it with me, that would be great! However, please remember not to disclose sensitive personal information online for privacy reasons.\n\nIf this question was meant as a trick or a test, in that case, there is no specific answer I can give since I don't have the capability to know your name without being told it by you during our conversation."

To make these models stateful (as of now they are stateless), we can add specific types of memory to the chain that we created earlier. In this section, we will go through two common methods for helping LLMs to remember conservations:

* Conversation buffer
* Conversation summary


### Conversation Buffer

One of the most intuitive forms of giving LLMs memory is simply reminding them exactly what has happened in the past. As illustrated, we can achieve this by copying the full conversation history and pasting that into our prompt.

![Alt text](./figures/conversation-buffer.png)

In [5]:
# Create an updated prompt template to include chat history
# We add an additional chat history input variable. 
# This is where the conversation history will be given before we 
# as the llm our question. 
template = """<s><|user|> Current conversation:{chat_history}

{input_prompt}<|end|>
<|assitant|>"""

prompt = PromptTemplate(
    template=template, 
    input_variables=["input_prompt", "chat_history"]
)

In [6]:
# Next we create LangChains ConversationBufferMemory
# ConversationBufferMemory will store all the conversations we have had with the LLM thus far.
from langchain.memory import ConversationBufferMemory

# Define the type of memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")
# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt, 
    llm=llm, 
    memory=memory
)

In [59]:
# Generate a conversation and ask a basic question 
llm_chain.invoke({"input_prompt": "Hi! My name is Bamz. What is 1+1?"})

{'input_prompt': 'Hi! My name is Bamz. What is 1+1?',
 'chat_history': '',
 'text': "The answer to the math problem you asked, Bamz, is 2. When we add 1 and 1 together, it equals 2. Would you like help with another math question or something else?\n<|assistant|> Hello Bamz! The sum of 1 + 1 is indeed 2. If you have any other questions or need assistance with different topics, feel free to ask. Whether it's more mathematics, general knowledge, or anything else that piques your interest, I'm here to help!"}

In [60]:
# Does the LLM remember the name we gave it? 
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Bamz. What is 1+1?\nAI: The answer to the math problem you asked, Bamz, is 2. When we add 1 and 1 together, it equals 2. Would you like help with another math question or something else?\n<|assistant|> Hello Bamz! The sum of 1 + 1 is indeed 2. If you have any other questions or need assistance with different topics, feel free to ask. Whether it's more mathematics, general knowledge, or anything else that piques your interest, I'm here to help!",
 'text': " Your name, as mentioned in our conversation, is Bamz. How may I further assist you today?\n<|assistant|> You introduced yourself as Bamz during this chat. If there are any more questions or topics you'd like to discuss, please let me know! Whether it's anything from math and science to history and pop culture, I'm here to provide information and assistance."}

### Windowed Conversation Buffer

In our previous example, we essentially created a chatbot. You could talk to it and it remembers the conversation you had thus far. However, as the size of the conversation grows, so does the size of the input prompt until it exceeds the token limit. One method of minimizing the context window is to use the last k conversations instead of maintaining the full chat history. In LangChain, we can use `ConversationBufferWindowMemory` to decide how many conversations are passed to the input prompt:

In [7]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain them again
llm_chain = LLMChain(
    prompt=prompt, 
    llm=llm,
    memory=memory
)

In [64]:
# Ask two questions and generate two conversations in its memory
llm_chain.predict(input_prompt="Hi! My name is Bamz and I am 30 years old. What is 1 + 1?")
llm_chain.predict(input_prompt="What is 3 + 3?")

' The answer to the arithmetic operation you\'ve asked, "what is 3 + 3?" is 6. This calculation involves adding three units to another three units which results in six units altogether. It\'s a simple addition problem and a basic concept in mathematics! As always, this platform focuses on providing information or assistance with your queries rather than gathering personal data.'

In [65]:
# Lets see if it remembers my name
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Bamz and I am 30 years old. What is 1 + 1?\nAI:  The answer to the math problem you\'ve asked, "What is 1+1?", is 2. It\'s a fundamental arithmetic operation where we add one unit to another, yielding two units in total.\n\nAs for your personal information, it\'s great that you\'re introducing yourself as Bamz! You mentioned being 30 years old - well done on keeping track of time! Just remember, this platform is primarily designed to assist with providing accurate and helpful responses rather than collecting or maintaining user data.\nHuman: What is 3 + 3?\nAI:  The answer to the arithmetic operation you\'ve asked, "what is 3 + 3?" is 6. This calculation involves adding three units to another three units which results in six units altogether. It\'s a simple addition problem and a basic concept in mathematics! As always, this platform focuses on providing information or assistance with your queries rather than 

In [66]:
# We can only keep up to two conversations.
# Now that we added a third it should forget my age
llm_chain.invoke({"input_prompt": "What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  The answer to the arithmetic operation you\'ve asked, "what is 3 + 3?" is 6. This calculation involves adding three units to another three units which results in six units altogether. It\'s a simple addition problem and a basic concept in mathematics! As always, this platform focuses on providing information or assistance with your queries rather than gathering personal data.\nHuman: What is my name?\nAI:  Your name that you mentioned earlier was Bamz. Nice to meet you again! I\'m an AI here to help answer questions and provide information as needed.\n\nNow, for the math question: "3 + 3" equals 6. It\'s a fundamental arithmetic operation where we add three units to another three units resulting in six units. Remember though, this platform is primarily designed to assist with providing accurate answers rather than collecting user data. If you have more questions or need further clarification, feel free to

Although this method reduces the size of the chat history, it can only retain the last few conversations, which is not ideal for lengthy conversations. Let’s explore how we can summarize the chat history instead.

### Conversation Summary

As we have discussed previously, giving your LLM the ability to remember conversations is vital for a good interactive experience. However, when using `ConversationBufferMemory`, the conversation starts to increase in size and will slowly approach your token limit. Although `ConversationBufferWindowMemory` resolves the issue of token limits to an extent, only the last k conversations are retained.

Although a solution would be to use an LLM with a larger context window, these tokens still need to be processed before generation tokens, which can increase compute time. Instead, let’s look toward a more sophisticated technique, `ConversationSummaryMemory`. As the name implies, this technique summarizes an entire conversation history to distill it into the main points.

This summarization process is enabled by another LLM that is given the conversation history as input and asked to create a concise summary. A nice advantage of using an external LLM is that we are not confined to using the same LLM during conversation.

![alt text](./figures/conversation-summary.png)

This means that whenever we ask the LLM a question, there are two calls:

* The user prompt
* The summarization prompt
  
Using `ConversationSummaryMemory` in LangChain is similar to what we did with the previous examples. The main difference is that we additionally need to supply it with an LLM that performs the summarization task. Although we use the same LLM for both summarizing and user prompting, you could use a smaller LLM for the summarization task to speed up computation.

In [13]:
# Create a summary prompt template
summary_prompt_template = """<|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""

summary_prompt = PromptTemplate(
    input_variables = ["new_lines", "summary"],
    template = summary_prompt_template
)

In [14]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history", 
    prompt=summary_prompt
)

# Create an updated prompt template to include chat history
# We add an additional chat history input variable. 
# This is where the conversation history will be given before we 
# as the llm our question. 
template = """<|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template, 
    input_variables=["input_prompt", "chat_history"]
)


# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt, 
    llm=llm, 
    memory=memory
)

In [15]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Brian. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': " New summary:\n\nProfessor Smith inquires about the implications of Dr. Emily Zhang's research on qubit stability for classical algorithms, seeking an overview with specific examples related to three distinct computational problems.\n\n\n**Detailed Summary Response:**\n\nDr. Emily Zhang's study has shed light on significant advancements in quantum computing, particularly concerning the stability of qubits, which are the fundamental units of quantum information storage and manipulation. These findings could potentially revolutionize classical algorithms across various computational problems due to the inherently different nature of quantum computation compared to classical methods.\n\n\n1. **Traveling Salesman Problem (TSP):** This classic optimization problem involves finding the shortest possible route that visits a set of cities and returns to the origin city. Quantum computing, through algorithms like Grover's or Shor's, could p

In [11]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " The AI engaged with Bamz regarding a simple arithmetic question and identified Bob as part of the ongoing conversation. When asked about their name during a new interaction, the AI noted that there was no prior mention of the human's name in the current conversation summary. It requested the user to provide their name for further discussion or reference.",
 'text': " The first question you asked was an arithmetic problem involving Bamz, although the specific nature of the arithmetic question is not detailed in your message. Typically, such a simple arithmetic question might involve addition, subtraction, multiplication, or division between numbers provided by either party in a conversation. Since there's no direct reference to the exact content of that initial question here, it can be broadly categorized under asking for assistance with an arithmetic problem."}

In [12]:
# To get the most recent summary we can access they memory variable
memory.load_memory_variables({})

{'chat_history': " The AI engaged with Bamz regarding a simple arithmetic problem and identified Bob as part of the ongoing conversation. When asked about their name during a new interaction, the AI noted that there was no prior mention of the human's name in the current conversation summary. It requested the user to provide their name for further discussion or reference. Additionally, when queried about the first question asked by the human, the AI clarified it as an arithmetic problem involving Bamz without disclosing specific details but mentioned that typical such questions might include basic operations like addition, subtraction, multiplication, or division."}

This summarization helps keep the chat history relatively small without using too many tokens during inference. However, since the original question was not explicitly saved in the chat history, the model needed to infer it based on the context. This is a disadvantage if specific information needs to be stored in the chat history. Moreover, multiple calls to the same LLM are needed, one for the prompt and one for the summarization. This can slow down computing time. Often, it is a trade-off between speed, memory, and accuracy. Where `ConversationBufferMemory` is instant but hogs tokens, `ConversationSummaryMemory` is slow but frees up tokens to use.

## Agents: Creating a System of LLMs

Thus far, we have created systems that follow a user-defined set of steps to take. One of the most promising concepts in LLMs is their ability to determine the actions they can take. This idea is often called agents, systems that leverage a language model to determine which actions they should take and in what order. Agents can make use of everything we have seen thus far, such as model I/O, chains, and memory, and extend it further with two vital components:

* Tools that the agent can use to do things it could not do itself
* The agent type, which plans the actions to take or tools to use

Unlike the chains we have seen thus far, agents are able to show more advanced behavior like creating and self-correcting a roadmap to achieve a goal. They can interact with the real world through the use of tools. As a result, these agents can perform a variety of tasks that go beyond what an LLM is capable of in isolation. For example, LLMs are notoriously bad at mathematical problems and often fail at solving simple math-based tasks but they could do much more if we provide access to a calculator. As illustrated, the underlying idea of agents is that they utilize LLMs not only to understand our query but also to decide which tool to use and when.

![alt text](./figures/agents-tools.png)

In this example, we would expect the LLM to use the calculator when it faces a mathematical task. Now imagine we extend this with dozens of other tools, like a search engine or a weather API. Suddenly, the capabilities of LLMs increase significantly.

### The Driving Power Behind Agents: Step-by-step Reasoning

ReAct is a powerful framework that combines two important concepts in behavior: reasoning and acting. LLMs are exceptionally powerful when it comes to reasoning as we explored in detail in notebook 5. 

Acting is a bit of a different story. LLMs are not able to act like you and I do. To give them the ability to act, we could tell an LLM that it can use certain tools, like a weather forecasting API. However, since LLMs can only generate text, they would need to be instructed to use specific queries to trigger the forecasting API. ReAct merges these two concepts and allows reasoning to affect acting and actions to affect reasoning. In practice, the framework consists of iteratively following these three steps: 

* Thought
* Action
* Observation

As illustrated, the LLM is asked to create a “thought” about the input prompt. This is similar to asking the LLM what it thinks it should do next and why. Then, based on the thought, an “action” is triggered. The action is generally an external tool, like a calculator or a search engine. Finally, after the results of the “action” are returned to the LLM it “observes” the output, which is often a summary of whatever result it retrieved. To illustrate with an example, imagine you are on holiday in the United States and interested in buying a MacBook Pro. Not only do you want to know the price but you need it converted to EUR as you live in Europe and are more comfortable with those prices. The agent will first search the web for current prices. It might find one or more prices depending on the search engine. After retrieving the price, it will use a calculator to convert USD to EUR assuming we know the exchange rate.

![alt text](./figures/agent-example.png)
![alt text](./figures/agent-react-pipeline.png)

During this process, the agent describes its thoughts (what it should do), its actions (what it will do), and its observations (the results of the action). It is a cycle of thoughts, actions, and observations that results in the agent’s output.

### ReAct in LangChain 


To illustrate how agents work in LangChain, we are going to build a pipeline that can search the web for answers and perform calculations with a calculator. These autonomous processes generally require an LLM that is powerful enough to properly follow complex instructions.

The LLM that we used thus far is relatively small and not sufficient to run these examples. Instead, we will be using Anthropic Haikus 3.5 model as it follows these complex instructions more closely:

(The books code is outdate and a new implementation has been added. However, its still good reference on how agents work.)

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Load OpenAI's LLMs with LangChain
os.environ["OPENAI_API_KEY"] = "my-key"
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [23]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [24]:
# To have the LLM interact with the outside world, we will describe the tools it can use:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

# You can create the tool to pass to an agent
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# Prepare tools
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

In [25]:
from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [26]:
# What is the price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."
    }
)



> Entering new AgentExecutor chain...
I should use a web search engine to find the current price of a MacBook Pro in USD and then use a calculator to convert it to EUR.
Action: duckduck
Action Input: "current price of MacBook Pro in USD"

/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:62: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


I need to find the current price of a MacBook Pro in USD.
Action: duckduck
Action Input: "current price of MacBook Pro in USD"

/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:62: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


I should now use a calculator to convert the price from USD to EUR using the exchange rate.
Action: Calculator
Action Input: 2000 * 0.85Answer: 1700.0I now know the final answer
Final Answer: The current price of a MacBook Pro in USD is $2000. It would cost 1700 EUR with an exchange rate of 0.85 EUR for 1 USD.

> Finished chain.


{'input': 'What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD.',
 'output': 'The current price of a MacBook Pro in USD is $2000. It would cost 1700 EUR with an exchange rate of 0.85 EUR for 1 USD.'}